<a href="https://colab.research.google.com/github/ShiYu0318/ShiYu-AI-Courses-Archive/blob/main/LLM_RAG_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai gradio pymupdf ragas

In [ ]:
import os
import fitz
import numpy as np
import gradio as gr
from google.colab import userdata
from google.colab import files
from google import genai
from google.genai import types


os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
client = genai.Client()
print("Client ready:", client is not None)

# TODO 1
## 串接 LLM API 生成回覆

In [ ]:
response = # TODO
print(response.text)

# TODO 2
## 包成可多輪對話的函式並用 Gradio 快速建立基本聊天介面

In [ ]:
def chat_with_gemini(message, history):
    """把對話歷史送進 Gemini，回傳新訊息。"""
    # 把 Gradio 的 history 格式轉成 Gemini 接受的格式
    contents = []
    for msg in history:
        role = "user" if msg["role"] == "user" else "model"
        contents.append({"role": role, "parts": [{"text": msg["content"]}]})
    contents.append({"role": "user", "parts": [{"text": message}]})

    response = # TODO
    return response.text

# 啟動 chatbot
gr.ChatInterface(
    # TODO
).launch(share=True, debug=False)

# TODO 3
## Chunk 分塊
請試著實作出 fixed_size chunking

In [ ]:
def extract_pdf(path: str) -> str:
    doc = fitz.open(path)
    return "\n".join(page.get_text() for page in doc)

def chunk_text(text: str, chunk_size: int = 100, overlap: int = 20) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        # TODO
    return [c for c in chunks if c.strip()]

# TODO 4
## Embedding 詞嵌入
使用 gemini-embedding-001 模型分別對 docs 和 query 做不同 task_type 的 Embedding
- RETRIEVAL_DOCUMENT
- RETRIEVAL_QUERY

In [ ]:
def embed_docs(texts: list[str]) -> np.ndarray:
    vecs = []
    for i, t in enumerate(texts):
        r = client.models.embed_content(
            # TODO
        )
        vecs.append(r.embeddings[0].values)
        if (i + 1) % 10 == 0:
            print(f"  embedded {i+1}/{len(texts)}...")
    return np.array(vecs)

def embed_query(text: str) -> np.ndarray:
    r = client.models.embed_content(
        # TODO
    )
    return np.array(r.embeddings[0].values)

# TODO 5
上傳檔案並呼叫函式

In [ ]:
# 把檔名改成你上傳的 PDF 名稱
pdf_path = "/content/.pdf"
print(f"Upload Finish：{pdf_path}\n")

raw_text = # TODO
chunks = # TODO
print(f"Extract Finish：{len(raw_text)} words, {len(chunks)} chunks\n")
print("Embedding...\n")

doc_embeddings = # TODO
print(f"\nEmbedding Finish：{doc_embeddings.shape}")

# TODO 6
RAG

In [ ]:
def retrieve(query: str, k: int = 3) -> list[str]:
    q_vec = # TODO
    q_norm = q_vec / np.linalg.norm(q_vec)
    d_norm = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
    sims   = d_norm @ q_norm
    top_k  = np.argsort(sims)[::-1][:k]
    return [chunks[i] for i in top_k]

def chat_with_rag(message: str, history: list) -> str:
    retrieved = # TODO
    context = "\n\n".join(f"[段落 {i+1}]\n{c}" for i, c in enumerate(retrieved))

    # 帶對話歷史的 contents
    contents = []
    for msg in history:
        role = "user" if msg["role"] == "user" else "model"
        contents.append({"role": role, "parts": [{"text": msg["content"]}]})
    contents.append({"role": "user", "parts": [{"text": message}]})

    response = client.models.generate_content(
        # TODO
    )
    return response.text

gr.ChatInterface(
    # TODO
).launch(share=True, debug=False)